# 00datacheck: what is inside the raw files?

Seoul subway performance dashboard, step 0.

This notebook only **reads and describes** the raw files in `data/raw`. It changes nothing and saves nothing.
For each of the four datasets it shows the columns, the size, the time span, the lines and stations covered, and any quirks.
The cleaning happens in the next notebook.

## Section 0: settings

In [1]:
from pathlib import Path

RAW = Path("data/raw")                      # <- folder with the raw files (relative to this notebook)
ENCODINGS = ["utf-8-sig", "cp949"]          # <- encodings to try, in this order
N_PREVIEW = 3                               # <- rows to show in each preview
N_TOP = 10                                  # <- length of the "top" lists

# Seoul Metro core lines (the ones that appear in every dataset)
CORE_LINES = [f"{i}호선" for i in range(1, 9)]   # <- 1호선 to 8호선

## Section 1: tools

In [2]:
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)


def read_csv_any(path, **kwargs):
    """Read a CSV, trying each encoding in ENCODINGS until one works."""
    for enc in ENCODINGS:
        try:
            df = pd.read_csv(path, encoding=enc, **kwargs)
            print(f"  read {path.name} as {enc}")
            return df, enc
        except UnicodeDecodeError:
            continue
    raise ValueError(f"No encoding in {ENCODINGS} worked for {path.name}")


import csv
import io
import re

RID_COLS = ["사용일자", "노선명", "역명", "승차총승객수", "하차총승객수", "등록일자"]
GLUED = re.compile(r'(\d")("\d{8}",")')   # end of one row stuck to the start of the next


def read_ridership(path):
    """Read a daily ridership file, repairing rows that were glued together without a line break."""
    raw = path.read_bytes()
    for enc in ENCODINGS:
        try:
            text = raw.decode(enc)
            break
        except UnicodeDecodeError:
            continue
    else:
        raise ValueError(f"No encoding in {ENCODINGS} worked for {path.name}")
    text, n_fixed = GLUED.subn(r"\1\n\2", text)
    # some months end each row with an extra empty field, so keep only the first 6 fields of each row
    rows = [r[:6] for r in csv.reader(io.StringIO(text.lstrip("\ufeff")))][1:]
    df = pd.DataFrame([r for r in rows if r], columns=RID_COLS, dtype=str)
    print(f"  read {path.name} as {enc}, repaired {n_fixed} glued row(s)")
    return df


def to_number(s):
    """Turn text like '3862.0' or ' 9.1 ' into a number (NaN if it is not one)."""
    return pd.to_numeric(s.astype(str).str.strip().replace({"": None, "nan": None}), errors="coerce")

## Section 2: daily ridership by line and station

Source: Seoul Open Data Plaza (OA-12914), transit card records.
One row per day, line and station: total boardings (승차) and alightings (하차).

In [3]:
files = sorted((RAW / "ridership").glob("*.csv"))
parts = []
for f in files:
    df = read_ridership(f)
    df["source_file"] = f.name
    parts.append(df)

rid = pd.concat(parts, ignore_index=True)

rid = rid.rename(columns={
    "사용일자": "date", "노선명": "line", "역명": "station",
    "승차총승객수": "board", "하차총승객수": "alight", "등록일자": "registered",
})
rid["date"] = pd.to_datetime(rid["date"], format="%Y%m%d")
rid["board"] = to_number(rid["board"])
rid["alight"] = to_number(rid["alight"])
rid.head(N_PREVIEW)

  read CARD_SUBWAY_MONTH_2019.csv as cp949, repaired 0 glued row(s)
  read CARD_SUBWAY_MONTH_2021.csv as cp949, repaired 1 glued row(s)
  read CARD_SUBWAY_MONTH_202609.csv as utf-8-sig, repaired 0 glued row(s)


,date,line,station,board,alight,registered,source_file
0,2019-01-01,2호선,을지로4가,3862.0,3728.0,20190104,CARD_SUBWAY_MONTH_2019.csv
1,2019-01-01,2호선,을지로3가,8104.0,7554.0,20190104,CARD_SUBWAY_MONTH_2019.csv
2,2019-01-01,2호선,을지로입구,22478.0,21330.0,20190104,CARD_SUBWAY_MONTH_2019.csv


In [4]:
print(f"rows: {len(rid):,}")
print(f"dates: {rid['date'].min():%Y-%m-%d} to {rid['date'].max():%Y-%m-%d}")
print(f"lines: {rid['line'].nunique()}, line and station pairs: {rid[['line','station']].drop_duplicates().shape[0]}")
print(f"missing numbers: board {rid['board'].isna().sum()}, alight {rid['alight'].isna().sum()}")
print(f"duplicate (date, line, station) rows: {rid.duplicated(['date','line','station']).sum()}")

rows: 453,537
dates: 2019-01-01 to 2026-09-30
lines: 27, line and station pairs: 646
missing numbers: board 0, alight 0
duplicate (date, line, station) rows: 152


In [5]:
# rows and passengers per file
summary = rid.groupby("source_file").agg(
    rows=("board", "size"),
    days=("date", "nunique"),
    lines=("line", "nunique"),
    stations=("station", "nunique"),
    boardings_million=("board", lambda s: round(s.sum() / 1e6, 1)),
)
summary["boardings_per_day_million"] = (summary["boardings_million"] / summary["days"]).round(2)
summary

,rows,days,lines,stations,boardings_million,boardings_per_day_million
source_file,,,,,,
CARD_SUBWAY_MONTH_2019.csv,215769,365,25,509,2716.7,7.44
CARD_SUBWAY_MONTH_2021.csv,219279,365,25,523,2007.8,5.50
CARD_SUBWAY_MONTH_202609.csv,18489,30,27,528,216.6,7.22


In [6]:
# which lines appear, and in which files (a 1 means present)
pd.crosstab(rid["line"], rid["source_file"]).clip(upper=1)

source_file,CARD_SUBWAY_MONTH_2019.csv,CARD_SUBWAY_MONTH_2021.csv,CARD_SUBWAY_MONTH_202609.csv
line,,,
1호선,1,1,1
2호선,1,1,1
3호선,1,1,1
4호선,1,1,1
5호선,1,1,1
6호선,1,1,1
7호선,1,1,1
8호선,1,1,1
9호선,1,1,1


In [7]:
# busiest stations by average daily boardings, per file
busiest = (rid.groupby(["source_file", "line", "station"])["board"].mean().round(0).reset_index().sort_values(["source_file", "board"], ascending=[True, False]))
busiest.groupby("source_file").head(N_TOP)

,source_file,line,station,board
10,CARD_SUBWAY_MONTH_2019.csv,2호선,강남,99816.0
52,CARD_SUBWAY_MONTH_2019.csv,2호선,잠실(송파구청),86772.0
59,CARD_SUBWAY_MONTH_2019.csv,2호선,홍대입구,80760.0
37,CARD_SUBWAY_MONTH_2019.csv,2호선,신림,70359.0
14,CARD_SUBWAY_MONTH_2019.csv,2호선,구로디지털단지,62872.0
63,CARD_SUBWAY_MONTH_2019.csv,3호선,고속터미널,61859.0
26,CARD_SUBWAY_MONTH_2019.csv,2호선,삼성(무역센터),59817.0
36,CARD_SUBWAY_MONTH_2019.csv,2호선,신도림,58900.0
2,CARD_SUBWAY_MONTH_2019.csv,1호선,서울역,57188.0
30,CARD_SUBWAY_MONTH_2019.csv,2호선,선릉,53999.0


## Section 3: hourly boardings by station (Seoul Metro, Lines 1 to 8)

Source: data.go.kr 15048032. One row per day, station and direction of the gate (승차 boarding, 하차 alighting), with 20 hourly columns.

In [8]:
f = next((RAW / "hourly").glob("*.csv"))
hr, enc = read_csv_any(f)
print(f"rows: {len(hr):,}")
print(f"columns: {list(hr.columns)}")
hr.head(N_PREVIEW)

  read 서울교통공사_역별 일별 시간대별 승하차인원 정보_20241231.csv as cp949
rows: 199,424
columns: ['연번', '수송일자', '호선', '역번호', '역명', '승하차구분', '06시이전', '06-07시간대', '07-08시간대', '08-09시간대', '09-10시간대', '10-11시간대', '11-12시간대', '12-13시간대', '13-14시간대', '14-15시간대', '15-16시간대', '16-17시간대', '17-18시간대', '18-19시간대', '19-20시간대', '20-21시간대', '21-22시간대', '22-23시간대', '23-24시간대', '24시이후']


,연번,수송일자,호선,역번호,역명,승하차구분,06시이전,06-07시간대,07-08시간대,08-09시간대,09-10시간대,10-11시간대,11-12시간대,12-13시간대,13-14시간대,14-15시간대,15-16시간대,16-17시간대,17-18시간대,18-19시간대,19-20시간대,20-21시간대,21-22시간대,22-23시간대,23-24시간대,24시이후
0,1,2024-01-01,1호선,150,서울역,승차,383,257,308,975,1528,1720,2243,2970,2713,1962,2716,2882,2871,2685,2922,2031,2279,1729,868,43
1,2,2024-01-01,1호선,150,서울역,하차,249,867,834,1201,1744,1731,2188,2707,2605,2350,2615,2501,2829,2095,1833,1465,1031,585,298,82
2,3,2024-01-01,1호선,151,시청,승차,188,92,167,245,322,357,503,596,589,714,935,1003,978,1116,951,909,723,462,176,3


In [9]:
hour_cols = [c for c in hr.columns if "시" in c and c not in ("수송일자",)]
hr["수송일자"] = pd.to_datetime(hr["수송일자"])
print(f"dates: {hr['수송일자'].min():%Y-%m-%d} to {hr['수송일자'].max():%Y-%m-%d} ({hr['수송일자'].nunique()} days)")
print(f"lines: {sorted(hr['호선'].unique())}")
print(f"stations (by code): {hr['역번호'].nunique()}, station names: {hr['역명'].nunique()}")
print(f"hour columns ({len(hour_cols)}): {hour_cols[0]} ... {hour_cols[-1]}")
print(f"gate types: {hr['승하차구분'].unique().tolist()}")
print(f"missing values in hour columns: {hr[hour_cols].isna().sum().sum()}")
print(f"duplicate (date, code, gate type) rows: {hr.duplicated(['수송일자','역번호','승하차구분']).sum()}")

dates: 2024-01-01 to 2024-12-31 (366 days)
lines: ['1호선', '2호선', '3호선', '4호선', '5호선', '6호선', '7호선', '8호선']
stations (by code): 274, station names: 241
hour columns (20): 06시이전 ... 24시이후
gate types: ['승차', '하차']
missing values in hour columns: 0
duplicate (date, code, gate type) rows: 0


In [10]:
# stations whose name maps to more than one code (transfer stations appear once per line)
codes_per_name = hr.groupby("역명")["역번호"].nunique()
print(f"names with more than one code: {(codes_per_name > 1).sum()}")
codes_per_name[codes_per_name > 1].head(N_TOP)

names with more than one code: 31


역명
가락시장              2
건대입구              2
고속터미널             2
공덕                2
교대(법원.검찰청)        2
군자(능동)            2
노원                2
대림(구로구청)          2
동대문               2
동대문역사문화공원(DDP)    3
Name: 역번호, dtype: int64

In [ ]:
# average weekday profile across the whole network (boardings only)
wk = hr[(hr["승하차구분"] == "승차") & (hr["수송일자"].dt.day_of_week < 5)]
profile = (wk.groupby("수송일자")[hour_cols].sum().mean().round(0)).astype(int)
profile.to_frame("avg weekday boardings, all stations")

C:\Users\likre\AppData\Local\Temp\ipykernel_36788\1476150810.py:2: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  wk = hr[(hr["승하차구분"] == "승차") & (hr["수송일자"].dt.dayofweek < 5)]


,"avg weekday boardings, all stations"
06시이전,66083
06-07시간대,140519
07-08시간대,381310
08-09시간대,461462
09-10시간대,268895
10-11시간대,186680
11-12시간대,184561
12-13시간대,200279
13-14시간대,356532
14-15시간대,215885


## Section 4: crowding by station, direction and half hour

Source: Seoul Open Data Plaza (OA-12928). Values are crowding percentages for trains leaving each station.
The two files describe the same thing in different layouts, so we read each one its own way and compare.

In [12]:
f23 = next((RAW / "crowding").glob("*20231231*.csv"))
c23, enc = read_csv_any(f23)
c23.columns = c23.columns.str.strip()
slot23 = [c for c in c23.columns if "시" in c and "분" in c]
c23[slot23] = c23[slot23].apply(to_number)
print(f"2023 file: {len(c23):,} rows, {len(slot23)} half-hour columns ({slot23[0]} ... {slot23[-1]})")
print(f"  day types: {c23['요일구분'].unique().tolist()}")
print(f"  lines: {sorted(c23['호선'].unique().tolist())}")
c23.head(N_PREVIEW)

  read 서울교통공사_지하철혼잡도정보_20231231.csv as cp949
2023 file: 1,668 rows, 39 half-hour columns (5시30분 ... 00시30분)
  day types: ['평일', '토요일', '일요일']
  lines: [1, 2, 3, 4, 5, 6, 7, 8]


,연번,요일구분,호선,역번호,출발역,상하구분,5시30분,6시00분,6시30분,7시00분,7시30분,8시00분,8시30분,9시00분,9시30분,10시00분,10시30분,11시00분,11시30분,12시00분,12시30분,13시00분,13시30분,14시00분,14시30분,15시00분,15시30분,16시00분,16시30분,17시00분,17시30분,18시00분,18시30분,19시00분,19시30분,20시00분,20시30분,21시00분,21시30분,22시00분,22시30분,23시00분,23시30분,00시00분,00시30분
0,1,평일,1,158,청량리,상선,9.1,7.5,4.9,6.9,11.2,15.7,14.0,10.5,17.0,11.2,13.2,15.1,13.3,15.6,17.5,19.6,16.9,20.3,21.8,19.0,28.5,28.4,29.4,32.6,42.4,64.6,60.4,40.5,25.4,25.5,20.7,27.8,24.4,27.2,27.6,19.6,14.3,4.9,NaN
1,2,평일,1,158,청량리,하선,20.4,18.1,26.8,41.8,66.6,66.4,54.7,47.0,38.2,31.1,28.1,17.0,20.8,23.1,21.9,23.3,25.8,16.6,21.8,25.4,20.6,24.0,29.5,27.9,23.6,27.8,17.1,12.1,11.0,10.2,8.9,10.2,9.2,8.3,7.6,10.8,7.2,0.3,0.2
2,3,평일,1,157,제기동,상선,9.0,9.2,6.0,8.0,13.1,18.4,17.8,13.9,21.9,15.6,15.3,17.9,17.4,17.3,20.9,24.2,20.8,24.7,25.5,26.9,28.5,31.1,30.0,40.4,50.8,70.9,68.3,45.7,28.4,28.0,22.9,33.2,27.8,29.9,30.0,20.6,14.5,8.3,NaN


In [13]:
f26 = next((RAW / "crowding").glob("*20260630*.xlsx"))
sheets = pd.read_excel(f26, sheet_name=None)
print(f"2026 file sheets: {list(sheets)}")
c26 = pd.concat(sheets.values(), ignore_index=True)
c26.columns = c26.columns.astype(str).str.strip()
slot26 = [c for c in c26.columns if "~" in c]
print(f"2026 file: {len(c26):,} rows, {len(slot26)} half-hour columns ({slot26[0]} ... {slot26[-1]})")
print(f"  day types: {c26['구분'].unique().tolist()}")
print(f"  lines: {sorted(c26['호선'].astype(str).unique().tolist())}")
c26.head(N_PREVIEW)

2026 file sheets: ['평일', '토요일', '일요일']
2026 file: 1,671 rows, 39 half-hour columns (05:30~06:00 ... 24:30~25:00)
  day types: ['평일', '토요일', '일요일']
  lines: ['1호선', '2호선', '3호선', '4호선', '5호선', '6호선', '7호선', '8호선']


,구분,호선,역번호,역명,상하구분,05:30~06:00,06:00~06:30,06:30~07:00,07:00~07:30,07:30~08:00,08:00~08:30,08:30~09:00,09:00~09:30,09:30~10:00,10:00~10:30,10:30~11:00,11:00~11:30,11:30~12:00,12:00~12:30,12:30~13:00,13:00~13:30,13:30~14:00,14:00~14:30,14:30~15:00,15:00~15:30,15:30~16:00,16:00~16:30,16:30~17:00,17:00~17:30,17:30~18:00,18:00~18:30,18:30~19:00,19:00~19:30,19:30~20:00,20:00~20:30,20:30~21:00,21:00~21:30,21:30~22:00,22:00~22:30,22:30~23:00,23:00~23:30,23:30~24:00,24:00~24:30,24:30~25:00
0,평일,1호선,150,서울역,상선,8.779913,22.453469,23.509790,38.342322,70.636331,73.774676,92.206309,57.983820,62.487310,39.496616,42.973773,32.980118,33.506768,33.714648,43.997462,38.551184,31.682379,31.142132,25.063452,27.785533,28.718274,26.876586,26.925308,29.668464,37.828363,45.464104,38.953046,29.146120,22.773749,20.260152,18.092217,20.260152,18.188905,20.429357,19.343274,14.847716,9.739848,6.514382,1.091371
1,평일,1호선,150,서울역,하선,11.768190,11.499577,18.510998,25.103109,40.975571,39.769987,23.861041,27.121102,24.029188,23.470812,19.220450,20.452623,26.025804,29.246193,29.461566,34.388748,37.700508,35.717005,32.988579,44.640440,56.814721,52.550761,59.211083,68.842005,79.102611,123.912255,68.397843,45.664657,40.145033,38.197970,37.834179,42.190355,37.670413,38.997462,28.857868,18.597716,12.868020,8.965736,0.000000
2,평일,1호선,151,시청,상선,7.332910,15.820643,19.614530,32.848985,48.279550,57.365165,57.893401,44.467005,48.094332,34.352792,39.548223,30.841189,31.173858,31.216461,40.406091,35.731810,29.069374,31.747039,25.843909,28.816171,29.596024,28.796229,30.757796,32.120699,42.097534,57.443800,52.425444,31.069162,25.768673,24.128596,22.343486,24.792724,21.352429,23.969966,22.991751,17.766497,11.164340,7.043147,1.104061


In [14]:
# most crowded station, direction and time on weekdays, in each file
def top_crowding(df, day_col, name_col, slots, label):
    long = df[df[day_col] == "평일"].melt(
        id_vars=["호선", name_col, "상하구분"], value_vars=slots, var_name="slot", value_name="crowding")
    out = long.sort_values("crowding", ascending=False).head(N_TOP).copy()
    out.insert(0, "file", label)
    return out.rename(columns={name_col: "station"})

pd.concat([
    top_crowding(c23, "요일구분", "출발역", slot23, "2023"),
    top_crowding(c26, "구분", "역명", slot26, "2026"),
]).round(1)

,file,호선,station,상하구분,slot,crowding
2997,2023,4,한성대입구,하선,8시00분,166.2
2995,2023,4,성신여대입구,하선,8시00분,165.6
2999,2023,4,혜화,하선,8시00분,158.1
2993,2023,4,길음,하선,8시00분,156.1
14411,2023,7,가산디지털단지,하선,18시00분,152.7
2691,2023,7,중곡,하선,7시30분,145.9
3413,2023,2,사당,외선,8시30분,144.0
3001,2023,4,동대문,하선,8시00분,143.2
2991,2023,4,미아사거리,하선,8시00분,143.2
2689,2023,7,용마산,하선,7시30분,142.4


In [15]:
# overall range of the values
for label, df, slots in [("2023", c23, slot23), ("2026", c26, slot26)]:
    v = df[slots].stack()
    print(f"{label}: min {v.min():.1f}, median {v.median():.1f}, max {v.max():.1f}, share of zeros {(v == 0).mean():.0%}")

2023: min 0.0, median 24.7, max 190.2, share of zeros 3%
2026: min 0.0, median 24.9, max 144.6, share of zeros 6%


## Section 5: train operations by line (2024)

Source: Seoul Open Data Plaza (OA-13318). A small table with measures as rows and lines as columns.
Some cells hold extra detail in brackets (branch lines, or the split between Seoul Metro and KORAIL trains), so we only look at it here and tidy it by hand later.

In [16]:
f = next((RAW / "operations").glob("*.csv"))
ops, enc = read_csv_any(f)
ops

  read 서울교통공사_열차운행현황_20240331.csv as cp949


,구분,1호선,2호선,3호선,4호선,5호선,6호선,7호선,8호선
0,구 간,서울~청량리,성수~성수[성수지선/신정지선],지축~오금,진접~남태령,방화~하남검단산/마천,응암순환~신내,장암~온수,암사~모란
1,영업거리(km),7.8,48.8[5.4/6.0],38.2,45.4,59.8,36.3,46.9,17.7
2,역 수 (개),10,43[4/3],34,29,56,39,42,18
3,소요시간(분),18,90[9/11],67.5,68,108.5,75,87,32.5
4,보유차량수(칸),160,750[20/24],490,520,696,312,616,120
5,열차편성수(1편성당 칸수),16(10),84(10[4/6]),49(10),52(10),87(8),39(8),77(8),20(6)
6,표정속도(km/h),26,32.5[36.0/32.7],34,40.1,33.1,29,32.3,32.7
7,열차주행거리(평일 km/일),5064.8,23728.4[1249.8/1344.2],14236.4,16019.6,20580.7,10724.4,18546.5,5193.8
8,운행 시격_RH(출근),3,2.5(7.0/10.0),3,2.5,2.5,4,2.5,4.5
9,운행 시격_NH(평시),5,5.5(10.0/10.0),6.5,5.5,6.5,8,6,8.5


## Section 6: how lines are named in each dataset

The datasets write the same line in different ways. This table shows the spelling used in each, for the core lines.

In [17]:
names = pd.DataFrame({"core line": CORE_LINES})
names["daily ridership"] = names["core line"].map(lambda x: x if x in set(rid["line"]) else "(missing)")
names["hourly"] = names["core line"].map(lambda x: x if x in set(hr["호선"].astype(str)) else "(other spelling)")
names["crowding 2023"] = names["core line"].map(lambda x: x.replace("호선", "") if x.replace("호선", "") in set(c23["호선"].astype(str)) else "(other spelling)")
names["crowding 2026"] = names["core line"].map(lambda x: x if x in set(c26["호선"].astype(str)) else "(other spelling)")
names["operations"] = names["core line"].map(lambda x: x if x in ops.columns else "(missing)")
names

,core line,daily ridership,hourly,crowding 2023,crowding 2026,operations
0,1호선,1호선,1호선,1,1호선,1호선
1,2호선,2호선,2호선,2,2호선,2호선
2,3호선,3호선,3호선,3,3호선,3호선
3,4호선,4호선,4호선,4,4호선,4호선
4,5호선,5호선,5호선,5,5호선,5호선
5,6호선,6호선,6호선,6,6호선,6호선
6,7호선,7호선,7호선,7,7호선,7호선
7,8호선,8호선,8호선,8,8호선,8호선


## Section 7: summary

Run all cells, then read the printed numbers above together with these notes:

- **Daily ridership** is the long time series (2019, 2021 and September 2026 in this sample). It covers more lines than Seoul Metro, has no station codes, and mixes encodings and number formats between years. The 2021 yearly file had one place where two rows were glued together (repaired by `read_ridership`). A few Gyeongui Line (경의선) stations have a second tiny row on some days (1 or 2 passengers); summing per day handles them.
- **COVID signal:** network boardings per day fell from about 7.4 million (2019) to 5.5 million (2021) and stand at 7.2 million in September 2026.
- **Hourly** covers Seoul Metro Lines 1 to 8 for all of 2024 and has station codes, so it is the natural source for a station list.
- **Crowding** has the same content in two layouts (2023 CSV, 2026 XLSX with three sheets). Line names and half-hour labels must be harmonised. The 2023 peak values reach 190% (Line 4 at 08:00) but 2026 peaks stop near 145%, so we should check whether the method changed before comparing years.
- **Operations** is a small summary table, best tidied by hand.
- Next notebook: `01clean`, which turns all four into tidy tables with one shared line and station lookup.

___
**END**